# E2 plots — feature × GNN matrix on Goodreads-Children

Reads `results/<DATASET>_long.csv` (produced by `scripts/aggregate_results.py`) and emits the four figures we need for the report:

1. **Heatmap** — feature × GNN, cell = mean test acc, annotated with ± std.
2. **Grouped bar chart** — one group per feature, two bars (GCN/SAGE), seed-std error bars.
3. **Per-seed scatter** — for each (feature, GNN), one dot per seed; shows variance honestly.
4. **Pairwise lift vs. `ogb`** — Δacc = feat − ogb, both GNNs side-by-side. This is what tells the H2 story (does TAPE lift the new dataset?).

All figures save to `results/figures/`.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Tweak only this when running locally vs. Colab vs. Drive
DATASET = 'goodreads_children'
RESULTS_DIR = Path('results')
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Canonical orderings (so plots are stable across n_seeds)
FEATURE_ORDER = ['ogb', 'TA', 'P', 'E', 'TA_P_E']
GNN_ORDER = ['GCN', 'SAGE']

long_csv = RESULTS_DIR / f'{DATASET}_long.csv'
if not long_csv.exists():
    sys.exit(f'{long_csv} not found. Run `python scripts/aggregate_results.py --dataset {DATASET}` first.')
df = pd.read_csv(long_csv)
df = df[(df['returncode'] == 0) & df['test_acc'].notna()].copy()
df['feature'] = pd.Categorical(df['feature'], FEATURE_ORDER, ordered=True)
df['gnn'] = pd.Categorical(df['gnn'], GNN_ORDER, ordered=True)
print(df[['feature','gnn','seed','test_acc']].sort_values(['feature','gnn','seed']))
n_seeds = df['seed'].nunique()
print(f'\nseeds present: {sorted(df.seed.unique())}  (n={n_seeds})')

## Figure 1 — feature × GNN heatmap

In [ ]:
agg = df.groupby(['feature','gnn'], observed=True)['test_acc'].agg(['mean','std','count']).reset_index()
mean_mat = agg.pivot(index='feature', columns='gnn', values='mean').reindex(FEATURE_ORDER)[GNN_ORDER]
std_mat = agg.pivot(index='feature', columns='gnn', values='std').reindex(FEATURE_ORDER)[GNN_ORDER]

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(mean_mat.values, cmap='YlGn', aspect='auto')
ax.set_xticks(range(len(GNN_ORDER))); ax.set_xticklabels(GNN_ORDER)
ax.set_yticks(range(len(FEATURE_ORDER))); ax.set_yticklabels(FEATURE_ORDER)
for i, feat in enumerate(FEATURE_ORDER):
    for j, gnn in enumerate(GNN_ORDER):
        m = mean_mat.values[i, j]
        s = std_mat.values[i, j]
        if pd.isna(m):
            txt = '—'
        elif pd.isna(s) or n_seeds <= 1:
            txt = f'{m:.3f}'
        else:
            txt = f'{m:.3f}\n±{s:.3f}'
        ax.text(j, i, txt, ha='center', va='center', fontsize=10)
ax.set_title(f'E2: test acc on {DATASET}  (n={n_seeds} seed{"s" if n_seeds!=1 else ""})')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='test acc')
plt.tight_layout()
plt.savefig(FIG_DIR / f'{DATASET}_heatmap.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / f'{DATASET}_heatmap.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — grouped bar chart with seed error bars

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
x = np.arange(len(FEATURE_ORDER))
width = 0.38
for k, gnn in enumerate(GNN_ORDER):
    means = [mean_mat.loc[f, gnn] if f in mean_mat.index else np.nan for f in FEATURE_ORDER]
    stds = [std_mat.loc[f, gnn] if f in std_mat.index else 0 for f in FEATURE_ORDER]
    stds = [0 if (pd.isna(s) or n_seeds <= 1) else s for s in stds]
    ax.bar(x + (k - 0.5) * width, means, width, yerr=stds, capsize=4, label=gnn)
ax.set_xticks(x); ax.set_xticklabels(FEATURE_ORDER)
ax.set_ylabel('test accuracy')
ax.set_title(f'E2 ({DATASET}): feature × GNN, n={n_seeds} seed{"s" if n_seeds!=1 else ""}')
ax.legend(title='GNN'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / f'{DATASET}_bars.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / f'{DATASET}_bars.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — per-seed scatter (honest about variance)

In [ ]:
fig, axes = plt.subplots(1, len(GNN_ORDER), figsize=(8, 4), sharey=True)
for ax, gnn in zip(axes, GNN_ORDER):
    sub = df[df['gnn'] == gnn]
    for j, feat in enumerate(FEATURE_ORDER):
        ys = sub.loc[sub['feature'] == feat, 'test_acc'].values
        if len(ys) == 0:
            continue
        xs = np.full_like(ys, j, dtype=float) + np.random.uniform(-0.08, 0.08, len(ys))
        ax.scatter(xs, ys, alpha=0.85)
        ax.hlines(np.mean(ys), j-0.25, j+0.25, colors='black', linewidth=2)
    ax.set_xticks(range(len(FEATURE_ORDER))); ax.set_xticklabels(FEATURE_ORDER, rotation=20)
    ax.set_title(gnn); ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('test accuracy')
fig.suptitle(f'E2 per-seed scatter ({DATASET})')
plt.tight_layout()
plt.savefig(FIG_DIR / f'{DATASET}_scatter.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / f'{DATASET}_scatter.pdf', bbox_inches='tight')
plt.show()

## Figure 4 — pairwise lift vs. `ogb` (the H2 picture)

For each (feature, GNN), compute `mean(test_acc[feature]) − mean(test_acc[ogb])` on the *same* GNN and seed set. Positive = TAPE features beat shallow features.

In [ ]:
# baseline = ogb mean per GNN
ogb_means = mean_mat.loc['ogb']
lift = mean_mat.subtract(ogb_means, axis=1)
lift = lift.drop(index='ogb')

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(len(lift.index))
width = 0.38
for k, gnn in enumerate(GNN_ORDER):
    ax.bar(x + (k - 0.5) * width, lift[gnn].values, width, label=gnn)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(lift.index)
ax.set_ylabel('test acc − ogb baseline')
ax.set_title(f'E2 lift over shallow features  ({DATASET}, n={n_seeds})')
ax.legend(title='GNN'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / f'{DATASET}_lift.png', dpi=200, bbox_inches='tight')
plt.savefig(FIG_DIR / f'{DATASET}_lift.pdf', bbox_inches='tight')
plt.show()
print(lift)